<a href="https://colab.research.google.com/github/Zuluke/Deep-Learning/blob/main/04_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><a href="https://www.nvidia.com/dli"> <img src="https://drive.google.com/uc?id=1J3bpGyJcz-7uOFkUNhvOBiReBCk-sUwR" width="200" </a></center>

# Assessment

In this assessment, you will train a new model that is able to recognize fresh and rotten fruit. You will need to get the model to a validation accuracy of `92%` in order to pass the assessment, though we challenge you to do even better if you can. You will have the use the skills that you learned in the previous exercises. Specifically, we suggest using some combination of transfer learning, data augmentation, and fine tuning. Once you have trained the model to be at least 92% accurate on the validation dataset, save your model, and then assess its accuracy. Let's get started!

## The Dataset

In this exercise, you will train a model to recognize fresh and rotten fruits. The dataset comes from [Kaggle](https://www.kaggle.com/sriramr/fruits-fresh-and-rotten-for-classification), a great place to go if you're interested in starting a project after this class. The dataset structure is in the `data/fruits` folder. There are 6 categories of fruits: fresh apples, fresh oranges, fresh bananas, rotten apples, rotten oranges, and rotten bananas. This will mean that your model will require an output layer of 6 neurons to do the categorization successfully. You'll also need to compile the model with `categorical_crossentropy`, as we have more than two categories.

In [1]:
path = "./Fundamentals_Deep_Learning/data/fruits/"

## Load ImageNet Base Model

We encourage you to start with a model pretrained on ImageNet. Load the model with the correct weights, set an input shape, and choose to remove the last layers of the model. Remember that images have three dimensions: a height, and width, and a number of channels. Because these pictures are in color, there will be three channels for red, green, and blue. We've filled in the input shape for you. This cannot be changed or the assessment will fail. If you need a reference for setting up the pretrained model, please take a look at [notebook 05b] where we implemented transfer learning.

In [2]:
from tensorflow import keras

base_model = keras.applications.VGG16(
    weights='imagenet',   
    input_shape=(224, 224, 3),
    include_top=False)

c:\Users\pichau\.conda\envs\deep-learning\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\pichau\.conda\envs\deep-learning\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


## Freeze Base Model

Next, we suggest freezing the base model, as done in [notebook 05b]. This is done so that all the learning from the ImageNet dataset does not get destroyed in the initial training.

In [3]:
# Freeze base model
base_model.trainable = False

## Add Layers to Model

Now it's time to add layers to the pretrained model. [Notebook 05b] can be used as a guide. Pay close attention to the last dense layer and make sure it has the correct number of neurons to classify the different types of fruit.

In [4]:
# Create inputs with correct shape
inputs = keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)

# Add pooling layer or flatten layer
x = keras.layers.GlobalAveragePooling2D()(x)  # Melhor que Flatten() para evitar overfitting
x = keras.layers.Dropout(0.2)(x)  # Regularização

# Add final dense layer
outputs = keras.layers.Dense(6, activation='softmax')(x)  # 6 classes de frutas

# Combine inputs and outputs to create model
model = keras.Model(inputs, outputs)

In [5]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │         3,078 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,717,766 (56.14 MB)

 Trainable params: 3,078 (12.02 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

## Compile Model

Now it's time to compile the model with loss and metrics options. Remember that we're training on a number of different categories, rather than a binary classification problem.

In [6]:
model.compile(loss='categorical_crossentropy', 
              metrics=['accuracy'])

## Augment the Data

If you'd like, try to augment the data to improve the dataset. Feel free to look at [notebook 04a] and [notebook 05b] for augmentation examples. There is also documentation for the [Keras ImageDataGenerator class](https://keras.io/api/preprocessing/image/#imagedatagenerator-class). This step is optional, but it may be helpful to get to 92% accuracy.

In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen_train = ImageDataGenerator(
    rescale=1.0/255, # Normalização dos pixels
    rotation_range=40, # Rotação aleatória
    width_shift_range=0.2, # Deslocamento horizontal
    height_shift_range=0.2, # Deslocamento vertical
    shear_range=0.2, # Cisalhamento
    zoom_range=0.2, # Zoom aleatório
    horizontal_flip=True, # Espelhamento horizontal
    fill_mode='nearest') # Preenchimento de pixels vazios

datagen_valid = ImageDataGenerator(rescale=1./255) # Apenas normalização para validação

## Load Dataset

Now it's time to load the train and validation datasets. Pick the right folders, as well as the right `target_size` of the images (it needs to match the height and width input of the model you've created). For a reference, check out [notebook 05b].

In [ ]:
# load and iterate training dataset
train_it = datagen_train.flow_from_directory(
    path+'train', # Caminho para os dados de treino
    target_size=(224, 224), # Mesmo tamanho do input do modelo
    color_mode="rgb",
    class_mode="categorical",
    batch_size=32)

# load and iterate validation dataset
valid_it = datagen_valid.flow_from_directory(
    path+'valid', # Caminho para os dados de validação
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=32)

Found 1182 images belonging to 6 classes.
Found 329 images belonging to 6 classes.


## Train the Model

Time to train the model! Pass the `train` and `valid` iterators into the `fit` function, as well as setting the desired number of epochs.

In [9]:
model.fit(train_it,
          validation_data=valid_it,
          steps_per_epoch=train_it.samples // train_it.batch_size,
          validation_steps=valid_it.samples // valid_it.batch_size,
          epochs=20)

Epoch 1/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 66s 2s/step - accuracy: 0.2583 - loss: 1.7292 - val_accuracy: 0.5063 - val_loss: 1.5161
Epoch 2/20
 1/36 ━━━━━━━━━━━━━━━━━━━━ 52s 2s/step - accuracy: 0.1875 - loss: 1.7047

c:\Users\pichau\.conda\envs\deep-learning\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


36/36 ━━━━━━━━━━━━━━━━━━━━ 17s 432ms/step - accuracy: 0.1875 - loss: 1.7047 - val_accuracy: 0.5188 - val_loss: 1.5154
Epoch 3/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 70s 2s/step - accuracy: 0.4304 - loss: 1.5309 - val_accuracy: 0.6438 - val_loss: 1.3457
Epoch 4/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 17s 437ms/step - accuracy: 0.5312 - loss: 1.4184 - val_accuracy: 0.6438 - val_loss: 1.3408
Epoch 5/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.5539 - loss: 1.4060 - val_accuracy: 0.5938 - val_loss: 1.2205
Epoch 6/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 17s 447ms/step - accuracy: 0.5625 - loss: 1.3240 - val_accuracy: 0.6125 - val_loss: 1.2167
Epoch 7/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 73s 2s/step - accuracy: 0.6130 - loss: 1.2691 - val_accuracy: 0.7063 - val_loss: 1.1025
Epoch 8/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 23s 605ms/step - accuracy: 0.7500 - loss: 1.1217 - val_accuracy: 0.7250 - val_loss: 1.1006
Epoch 9/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 134s 4s/step - accuracy: 0.6443 - loss: 1.1824 - val_accuracy: 0.7688 - val_los

## Unfreeze Model for Fine Tuning

If you have reached 92% validation accuracy already, this next step is optional. If not, we suggest fine tuning the model with a very low learning rate.

In [10]:
# Unfreeze the base model
base_model.trainable = True

# Compile the model with a low learning rate
model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-5), # Learning rate baixo
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

In [11]:
model.fit(train_it,
          validation_data=valid_it,
          steps_per_epoch=train_it.samples // train_it.batch_size,
          validation_steps=valid_it.samples // valid_it.batch_size,
          epochs=10) # Mais 10 épocas com learning rate baixo

Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 224s 6s/step - accuracy: 0.8191 - loss: 0.5309 - val_accuracy: 0.9219 - val_loss: 0.2874
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 21s 445ms/step - accuracy: 0.9062 - loss: 0.3964 - val_accuracy: 0.8813 - val_loss: 0.3040
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 228s 6s/step - accuracy: 0.8861 - loss: 0.3297 - val_accuracy: 0.9312 - val_loss: 0.1782
Epoch 4/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 21s 450ms/step - accuracy: 0.9375 - loss: 0.1678 - val_accuracy: 0.9406 - val_loss: 0.1810
Epoch 5/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 227s 6s/step - accuracy: 0.9183 - loss: 0.2291 - val_accuracy: 0.9062 - val_loss: 0.2163
Epoch 6/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 21s 443ms/step - accuracy: 0.9688 - loss: 0.1572 - val_accuracy: 0.9469 - val_loss: 0.1488
Epoch 7/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 228s 6s/step - accuracy: 0.9383 - loss: 0.1770 - val_accuracy: 0.9594 - val_loss: 0.1272
Epoch 8/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 22s 449ms/step - accuracy: 0.9375 - loss: 0.1949 - val_accuracy: 0.

## Evaluate the Model

Hopefully, you now have a model that has a validation accuracy of 92% or higher. If not, you may want to go back and either run more epochs of training, or adjust your data augmentation.

Once you are satisfied with the validation accuracy, evaluate the model by executing the following cell. The evaluate function will return a tuple, where the first value is your loss, and the second value is your accuracy. To pass, the model will need have an accuracy value of `92% or higher`.

In [ ]:
model.evaluate(valid_it, steps=valid_it.samples//valid_it.batch_size)

10/10 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.9531 - loss: 0.1195


[0.11950705200433731, 0.953125]